# 13 Piezometer locations

<!-- navigation -->
[<kbd>← 12 Download river levels (Rijkswaterstaat)</kbd>](12%20Download%20river%20levels%20%28Rijkswaterstaat%29.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>15 Overview of measurements →</kbd>](15%20Overview%20of%20measurements.ipynb)

**Required** · minimal path: 00 → 10 → **13** → 20 → 51 → 52

Reads and checks the piezometer table, and writes the piezometers to the project geopackage, with a map per transect.

| | |
|---|---|
| **Input** | `piezometers.xlsx` (template; notebook 11 adds the downloaded filters), cross-section line in the transects metadata (for the map) |
| **Output** | `project_data.gpkg`, layer `piezometers` |

One row per piezometer filter with its transect, coordinates and filter screen, and three columns for the model that you fill in yourself: `x_crosssection`, `modellayer` and `use_calibration` (see section 2).

This notebook comes after the time series (10, 11, 12), because notebook 11 adds the downloaded BRO filters to `piezometers.xlsx`, with `modellayer` and `x_crosssection` still empty. Fill them in, run this notebook and check the table and the maps.

In [ ]:
from pathlib import Path

import timflow_levee_toolkit as tlt
tlt.show_versions()

In [ ]:
## UPDATE FOR YOUR PROJECT: the project folder (with project_settings.toml and the data)
project_dir = Path.cwd().parent / "examples" / "papendrecht"

project = tlt.load_project(project_dir)
units = project.units
project

## 1. Read the piezometers

In [ ]:
piezometers = tlt.read_piezometer_table(project.general_file("piezometers"))
piezometers

## 2. Check `x_crosssection`, `modellayer` and `use_calibration`
These three columns are not in the downloaded data: fill them in yourself in `piezometers.xlsx`, for each filter.

| Column | What to fill in | How |
|---|---|---|
| `x_crosssection` | x-coordinate of the filter in the model [m]: the distance from the boundary between foreland and hinterland (usually the inner toe of the levee), **positive towards the hinterland**, negative in the foreland and the river | Measure it along the cross-section line, e.g. in QGIS with the measuring tool from the inner toe, or project the filter perpendicularly onto the line. The map in section 4 shows the line (from the river side) and the values that are filled in, so you can check them. |
| `modellayer` | the model aquifer the filter screen is in: **0 = the upper aquifer**, 1 = the next one, ... | Compare `screen_top` and `screen_bottom` with the aquifers of the geohydrological scheme (`model_parameters_<transect>.xlsx`, notebook 20; CPTs and boreholes in notebook 21). |
| `use_calibration` | 1 (default, also when empty): use the filter in the calibration; 0: only show it in the figures | Use 0 for a filter that is not in an aquifer of the model, e.g. in the levee body or in the semi-confining top layer, or with unreliable data. Give the reason in `remark`. A filter you don't want to see at all can be deleted from the table. |

A filter without `x_crosssection` or `modellayer` can't be used in the model (notebooks 51 and 52). After filling them in, run this notebook again.

In [ ]:
from IPython.display import HTML, display

incomplete = tlt.check_piezometer_table(piezometers)
if incomplete.empty:
    print(f"All {len(piezometers)} filters have x_crosssection and modellayer")
else:
    display(HTML(
        '<div style="background-color: #f8d7da; color: #842029; border: 1px solid #f5c2c7; '
        'border-radius: 4px; padding: 10px 14px;">'
        f'<b>{len(incomplete)} of {len(piezometers)} filters are not complete.</b> Fill in '
        '<code>x_crosssection</code> and <code>modellayer</code> (or correct them) in '
        f'<code>{project.relpath(project.general_file("piezometers"))}</code>, as explained above, '
        'and run this notebook again. Until then these filters can\'t be used in the model.</div>'
    ))
incomplete

In [ ]:
piezometers.sort_values(["transect", "x_crosssection", "modellayer"])[
    ["name", "transect", "x_crosssection", "modellayer", "screen_top", "screen_bottom", "use_calibration", "remark"]
]

## 3. Save to the project geopackage

In [ ]:
gdf = tlt.write_piezometers(
    piezometers,
    project.general_file("project_data"),
    layer=project.settings["paths"]["piezometer_layer"],
    crs=project.crs,
)
print(f"{len(gdf)} piezometers written to {project.relpath(project.general_file('project_data'))}")

## 4. Map per transect
Cross-section line (from the river side) and the piezometers with their `x_crosssection`.

In [ ]:
for transect, group in gdf.groupby("transect"):
    f, ax = tlt.plot_transect_map(group, line=project.crosssection(transect), crs=project.crs)  # source="brt": PDOK
    tlt.finalize_and_save(f, project.transect_dir(transect, "intermediate"), f"{project.name} - {transect}", tag="piezometers_map", file_label=transect)

---

<!-- navigation -->
[<kbd>← 12 Download river levels (Rijkswaterstaat)</kbd>](12%20Download%20river%20levels%20%28Rijkswaterstaat%29.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>15 Overview of measurements →</kbd>](15%20Overview%20of%20measurements.ipynb)